# Road Accident Analysis — Data Cleaning
**Capstone Project | Sahil Pradhan | Roll: 2330043**

This notebook handles all data preprocessing steps before loading into Power BI.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

print('Libraries loaded successfully.')

## Step 1: Load Raw Datasets

In [ ]:
# Load main accident records
df_accidents = pd.read_csv('accident_records.csv')
df_roads     = pd.read_csv('road_type_summary.csv')
df_states    = pd.read_csv('state_accident_index.csv')

print('accident_records shape    :', df_accidents.shape)
print('road_type_summary shape   :', df_roads.shape)
print('state_accident_index shape:', df_states.shape)

## Step 2: Inspect Data Quality

In [ ]:
print('=== Missing Values ===')
print(df_accidents.isnull().sum())
print('\n=== Data Types ===')
print(df_accidents.dtypes)
print('\n=== Sample Rows ===')
df_accidents.head()

## Step 3: Handle Missing Values

In [ ]:
# Fill missing numeric fields with median
num_cols = df_accidents.select_dtypes(include=[np.number]).columns
df_accidents[num_cols] = df_accidents[num_cols].fillna(df_accidents[num_cols].median())

# Fill missing categorical fields with mode
cat_cols = df_accidents.select_dtypes(include=['object']).columns
for col in cat_cols:
    df_accidents[col] = df_accidents[col].fillna(df_accidents[col].mode()[0])

print('Missing values after cleaning:')
print(df_accidents.isnull().sum().sum())

## Step 4: Standardise and Derive Columns

In [ ]:
# Standardise column names
df_accidents.columns = [c.strip().lower().replace(' ', '_') for c in df_accidents.columns]

# Convert date column
if 'accident_date' in df_accidents.columns:
    df_accidents['accident_date'] = pd.to_datetime(df_accidents['accident_date'], errors='coerce')
    df_accidents['year']  = df_accidents['accident_date'].dt.year
    df_accidents['month'] = df_accidents['accident_date'].dt.month
    df_accidents['hour']  = df_accidents['accident_date'].dt.hour

# Severity score (0-1 normalised)
if 'casualties' in df_accidents.columns and 'fatalities' in df_accidents.columns:
    max_c = df_accidents['casualties'].max()
    max_f = df_accidents['fatalities'].max()
    df_accidents['severity_score'] = (
        0.6 * df_accidents['fatalities'] / max_f +
        0.4 * df_accidents['casualties'] / max_c
    ).round(3)

print('Derived columns added. Shape:', df_accidents.shape)

## Step 5: Remove Duplicates & Outliers

In [ ]:
before = len(df_accidents)
df_accidents.drop_duplicates(inplace=True)
print(f'Removed duplicates: {before - len(df_accidents)} rows')

# Remove extreme outliers using IQR
for col in ['casualties', 'fatalities']:
    if col in df_accidents.columns:
        Q1, Q3 = df_accidents[col].quantile([0.25, 0.75])
        IQR = Q3 - Q1
        df_accidents = df_accidents[
            (df_accidents[col] >= Q1 - 3*IQR) &
            (df_accidents[col] <= Q3 + 3*IQR)
        ]
print('Final clean dataset shape:', df_accidents.shape)

## Step 6: Exploratory Visualisations

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

# Accidents by year
if 'year' in df_accidents.columns:
    df_accidents.groupby('year')['fatalities'].sum().plot(
        kind='bar', ax=axes[0], color='#C0392B', edgecolor='black')
    axes[0].set_title('Total Fatalities per Year')
    axes[0].set_xlabel('Year')
    axes[0].set_ylabel('Fatalities')

# Accidents by road type
if 'road_type' in df_accidents.columns:
    df_accidents['road_type'].value_counts().plot(
        kind='pie', ax=axes[1], autopct='%1.1f%%',
        colors=['#E74C3C','#E67E22','#F1C40F','#27AE60','#2980B9'])
    axes[1].set_title('Accident Distribution by Road Type')

plt.tight_layout()
plt.savefig('eda_preview.png', dpi=150)
plt.show()
print('EDA chart saved.')

## Step 7: Export Clean CSVs for Power BI

In [ ]:
df_accidents.to_csv('accident_records_clean.csv', index=False)
print('Clean files exported:')
print('  accident_records_clean.csv')
print('\nData cleaning complete!')